# 01 Raw data inventory

目标：用真实本地 CSV 验证文件、原始表头、必需字段、日期可解析性、单位及主键/一对多风险。

- 输入：`Data_processing/manual_processing/*.csv`、`config/schemas.yaml`
- 输出：`data/audit/raw_file_inventory.csv`、`raw_schema_inventory.json`、`missing_required_fields.csv`、预检与状态文档
- 主键：各表由 `schemas.yaml` 单独声明
- 时间范围：读取真实覆盖；smoke 模式每表读取前1000行并检查末行
- 前置依赖：00
- 本 Notebook 不负责：清洗或写回任何原始 CSV

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import csv
from datetime import datetime, timezone
import polars as pl
from src.macro_data import read_tabular_header, read_tabular_sample
from src.schema_utils import validate_raw_schema

SCHEMAS = CONFIG["schemas"]
inventory_rows = []
schema_rows = []
missing_rows = []
samples = {}

for table_name, schema in SCHEMAS.items():
    if table_name in {"version", "note", "processed_outputs"} or not isinstance(schema, dict):
        continue
    path = PATHS[schema["path_key"]]
    exists = path.is_file()
    header = read_tabular_header(path, schema) if exists else []
    validation = validate_raw_schema(header, schema, path.name) if exists else {
        "source": path.name,
        "missing_required_columns": list(schema.get("required_columns", [])),
        "forbidden_columns_present": [],
        "valid": False,
    }
    for column in validation["missing_required_columns"]:
        missing_rows.append({
            "table": table_name,
            "file": path.name,
            "missing_column": column,
            "affected_stage": schema["path_key"],
        })
    sample = None
    sample_error = None
    if exists:
        try:
            if str(schema.get("file_format", path.suffix.lstrip("."))).lower() in {"xlsx", "xls"}:
                sample = pl.from_pandas(
                    read_tabular_sample(
                        path,
                        schema,
                        n_rows=int(PARAMS["smoke_rows_per_file"]),
                    )
                )
            else:
                sample = pl.read_csv(
                    path,
                    n_rows=int(PARAMS["smoke_rows_per_file"]),
                    infer_schema=False,
                    null_values=["", "NA", "N/A", "NULL", "null"],
                    encoding="utf8-lossy",
                )
        except Exception as error:
            sample_error = f"{type(error).__name__}: {error}"
    samples[table_name] = sample
    inventory_rows.append({
        "table": table_name,
        "file": path.name,
        "path": str(path.relative_to(ROOT)),
        "exists": exists,
        "bytes": path.stat().st_size if exists else None,
        "modified_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat() if exists else None,
        "column_count": len(header),
        "sample_rows": sample.height if sample is not None else 0,
        "sample_read_error": sample_error,
    })
    schema_rows.append({
        "table": table_name,
        "file": path.name,
        "actual_columns": header,
        "required_columns": schema.get("required_columns", []),
        "rename_columns": schema.get("rename_columns", {}),
        "date_columns": schema.get("date_columns", []),
        "primary_key": schema.get("primary_key", []),
        "missing_required_columns": validation["missing_required_columns"],
        "valid": validation["valid"] and sample_error is None,
    })

inventory = pl.DataFrame(inventory_rows)
missing_required = pl.DataFrame(
    missing_rows,
    schema={"table": pl.String, "file": pl.String, "missing_column": pl.String, "affected_stage": pl.String},
)
print(inventory)
assert inventory["exists"].all()
assert inventory["sample_read_error"].null_count() == inventory.height


In [ ]:
date_checks = []
key_checks = []
for table_name, schema in SCHEMAS.items():
    if table_name in {"version", "note", "processed_outputs"} or not isinstance(schema, dict):
        continue
    sample = samples.get(table_name)
    if sample is None:
        continue
    for raw_date in schema.get("date_columns", []):
        if raw_date not in sample.columns:
            # renamed FF date fields are still the raw empty-name column during inventory.
            reverse = {v: k for k, v in schema.get("rename_columns", {}).items()}
            raw_date = reverse.get(raw_date, raw_date)
        if raw_date not in sample.columns:
            continue
        as_text = sample[raw_date].cast(pl.String).str.strip_chars()
        if raw_date in {"", "yyyymm", "yearmo"}:
            parsed = as_text.str.strptime(pl.Date, "%Y%m", strict=False)
        elif raw_date == "yyyymmdd":
            parsed = as_text.str.strptime(pl.Date, "%Y%m%d", strict=False)
        else:
            parsed = as_text.str.to_date(strict=False)
        nonmissing = int(as_text.is_not_null().sum())
        parsed_count = int(parsed.is_not_null().sum())
        date_checks.append({
            "table": table_name,
            "column": raw_date or "<unnamed_first_column>",
            "sample_nonmissing": nonmissing,
            "sample_parsed": parsed_count,
            "parse_rate": parsed_count / nonmissing if nonmissing else None,
            "sample_min": str(parsed.min()) if parsed_count else None,
            "sample_max": str(parsed.max()) if parsed_count else None,
        })
    keys = schema.get("primary_key", [])
    if keys and all(key in sample.columns for key in keys):
        duplicated_rows = int(sample.select(keys).is_duplicated().sum())
        key_checks.append({
            "table": table_name,
            "keys": "|".join(keys),
            "sample_rows": sample.height,
            "sample_duplicate_rows": duplicated_rows,
            "duplicate_risk": duplicated_rows > 0,
        })

date_checks_df = pl.DataFrame(date_checks)
key_checks_df = pl.DataFrame(key_checks)
print(date_checks_df)
print(key_checks_df)


In [ ]:
# Locked special-field checks.
required_special = {
    "distribution_information": {"DisType", "DisPaymentType", "DisDetailType"},
    "monthly_stock_file": {
        "ShareType", "SecurityType", "SecuritySubType", "USIncFlg", "IssuerType",
        "PrimaryExch", "ConditionalType", "TradingStatusFlg", "MthRetFlg",
    },
    "mthadjustfactors": {"mthcumfacshr"},
    "ccm_linktable": {"LPERMNO"},
}
for table_name, columns in required_special.items():
    actual = set(next(row["actual_columns"] for row in schema_rows if row["table"] == table_name))
    assert columns <= actual, (table_name, columns - actual)

ff_unit_checks = []
for table_name, factor_columns in {
    "ff_monthly": ["mktrf", "smb", "hml", "rf"],
    "mom_monthly": ["mom"],
    "ff_daily": ["mktrf", "smb", "hml", "rf"],
}.items():
    sample = samples[table_name]
    numeric = sample.select([
        pl.col(column).cast(pl.String).str.strip_chars().cast(pl.Float64, strict=False).alias(column)
        for column in factor_columns
    ])
    maximum = max(float(numeric[column].abs().max()) for column in factor_columns)
    ff_unit_checks.append({
        "table": table_name,
        "sample_max_abs_raw": maximum,
        "detected_unit": "percent" if maximum > 1 else "decimal",
        "conversion": "divide_by_100" if maximum > 1 else "none",
    })
ff_unit_df = pl.DataFrame(ff_unit_checks)
assert set(ff_unit_df["detected_unit"].to_list()) == {"percent"}
print(ff_unit_df)


In [ ]:
# Macro-source checks use the real files and preserve missing values.
import pandas as pd
from src.macro_data import build_monthly_macro_features

macro_preview, macro_audits = build_monthly_macro_features(
    PATHS["sentiment"],
    PATHS["cfnai"],
    sentiment_sheet=CONFIG["path_options"]["sentiment"]["sheet_name"],
)
sentiment_schema = SCHEMAS["sentiment"]
cfnai_schema = SCHEMAS["cfnai"]
assert sentiment_schema["required_columns"] == ["yearmo", "SENT"]
assert sentiment_schema["sheet_name"] == "DATA"
assert cfnai_schema["required_columns"] == ["observation_date", "cfnai"]
coverage = macro_audits["coverage"].set_index("source")
assert pd.Timestamp(coverage.loc["Sentiment", "observation_month_min"]) == pd.Timestamp("1958-01-01")
assert pd.Timestamp(coverage.loc["Sentiment", "observation_month_max"]) == pd.Timestamp("2025-12-01")
assert pd.Timestamp(coverage.loc["CFNAI", "observation_month_min"]) == pd.Timestamp("1980-01-01")
assert pd.Timestamp(coverage.loc["CFNAI", "observation_month_max"]) == pd.Timestamp("2025-12-01")
assert macro_audits["duplicate_months"].empty
assert "SENT_ORTH" not in macro_preview.columns


In [ ]:
# Sample-level one-to-many risk diagnostics; later cleaning notebooks perform full-key exclusions.
one_to_many = []
crsp_link = samples["crsp_fundno_wficn"]
one_to_many.append({
    "relationship": "crsp_fundno->wficn",
    "sample_conflict_keys": crsp_link.group_by("crsp_fundno").agg(pl.col("wficn").n_unique().alias("n")).filter(pl.col("n") > 1).height,
})
s12_link = samples["fundno_wficn"]
one_to_many.append({
    "relationship": "fundno+fdate+rdate->wficn",
    "sample_conflict_keys": s12_link.group_by(["fundno", "fdate", "rdate"]).agg(pl.col("wficn").n_unique().alias("n")).filter(pl.col("n") > 1).height,
})
monthly_stock = samples["monthly_stock_file"].with_columns(
    pl.col("CUSIP").cast(pl.String).str.slice(0, 8).str.zfill(8).alias("cusip8"),
    pl.col("MthCalDt").cast(pl.String).str.to_date(strict=False).dt.strftime("%Y-%m").alias("month"),
)
one_to_many.append({
    "relationship": "cusip8+month->PERMNO",
    "sample_conflict_keys": monthly_stock.group_by(["cusip8", "month"]).agg(pl.col("PERMNO").n_unique().alias("n")).filter(pl.col("n") > 1).height,
})
one_to_many_df = pl.DataFrame(one_to_many)
print(one_to_many_df)


In [ ]:
from src.atomic_write import write_csv_atomic, write_json_atomic, write_text_atomic
from src.runtime_paths import active_release_root, clean_output_path, output_path

if not SMOKE:
    from src.full_run_inventory import run_full_inventory_audit
    full_dates, full_keys, full_risks, full_row_counts = run_full_inventory_audit(
        SCHEMAS,
        PATHS,
        temp_directory=PATHS["full_run_tmp"],
        memory_limit=str(PARAMS["duckdb_memory_limit"]),
    )
    date_checks_df = pl.from_pandas(full_dates)
    key_checks_df = pl.from_pandas(full_keys)
    one_to_many_df = pl.from_pandas(full_risks)
    inventory = inventory.with_columns(
        pl.col("file").replace_strict(full_row_counts, default=None).cast(pl.Int64).alias("full_rows")
    )

release_root = active_release_root(ROOT)
audit_dir = output_path(
    ROOT, RUN_MODE, "01_raw_data_inventory", "data/audit/raw_file_inventory.csv"
).parent
docs_dir = (
    clean_output_path(ROOT, "docs")
    if EXECUTION_CONTEXT == "clean_rebuild_compute"
    else (release_root / "docs" if release_root is not None else ROOT / "docs")
)
inventory_path = write_csv_atomic(inventory, audit_dir / "raw_file_inventory.csv")
schema_path = write_json_atomic(audit_dir / "raw_schema_inventory.json", schema_rows)
missing_path = write_csv_atomic(missing_required, audit_dir / "missing_required_fields.csv")
write_csv_atomic(date_checks_df, audit_dir / "01_date_parse_checks.csv")
write_csv_atomic(key_checks_df, audit_dir / "01_sample_primary_key_checks.csv")
write_csv_atomic(one_to_many_df, audit_dir / "01_sample_one_to_many_risks.csv")
write_csv_atomic(ff_unit_df, audit_dir / "01_fama_french_unit_checks.csv")
write_csv_atomic(pl.from_pandas(macro_audits["coverage"]), audit_dir / "01_macro_source_coverage.csv")

blocking_count = missing_required.height + int((inventory["sample_read_error"].is_not_null()).sum())
preflight = f"""# PREFLIGHT REPORT

Generated from the configured real CSV and Excel inputs in `Data_processing/manual_processing/`.

- Files checked: {inventory.height}
- Missing files: {int((~inventory["exists"]).sum())}
- Missing required fields: {missing_required.height}
- Sample read errors: {int(inventory["sample_read_error"].is_not_null().sum())}
- Current input blocking count: {blocking_count}
- Fama-French detected unit: percent; conversion is division by 100 exactly once.
- `mthcumfacshr`: present.
- CCM `LPERMNO`: present.
- Distribution classification fields: present.
- CIZ common-stock universe fields and `MthRetFlg`: present.
- Sentiment DATA sheet: 1958-01 to 2025-12; duplicate months: 0; `SENT_ORTH` not selected.
- CFNAI current-vintage file: 1980-01 to 2025-12; duplicate months: 0.
- Macro data are observation-month/current-vintage series; no historical real-time availability is claimed.

{"Full mode used out-of-core complete-file row/date/key scans; the bounded preview is display-only." if not SMOKE else f"The smoke inventory reads the first {PARAMS['smoke_rows_per_file']} real rows of every CSV."}
Conflict resolution and exact exclusions are performed by the corresponding numbered notebooks.
No raw CSV or Excel input was modified.
"""
write_text_atomic(docs_dir / "PREFLIGHT_REPORT.md", preflight)

blockers = "# IMPLEMENTATION BLOCKERS\n\n"
if blocking_count == 0:
    blockers += "No active input-file or required-field blockers were found by Notebook 01.\n"
else:
    blockers += missing_required.write_csv()
write_text_atomic(docs_dir / "IMPLEMENTATION_BLOCKERS.md", blockers)

status = f"""# IMPLEMENTATION STATUS

| Notebook | Implementation | Static check | Restart run | Real-data smoke test |
|---|---|---|---|---|
| 00_environment_and_config | complete | passed | passed | passed |
| 01_raw_data_inventory | complete | passed | passed | {'passed' if blocking_count == 0 else 'blocked'} |

Current blocking count: {blocking_count}.
"""
write_text_atomic(docs_dir / "IMPLEMENTATION_STATUS.md", status)
print(preflight)


In [ ]:
from src.audit_utils import write_run_summary

summary_path = write_run_summary(
    ROOT,
    "01_raw_data_inventory",
    stage="01",
    started_at=STARTED_AT,
    input_files=[row["path"] for row in inventory_rows],
    output_files=[
        str(inventory_path.relative_to(ROOT)),
        str(schema_path.relative_to(ROOT)),
        str(missing_path.relative_to(ROOT)),
        str((docs_dir / "PREFLIGHT_REPORT.md").relative_to(ROOT)),
        str((docs_dir / "IMPLEMENTATION_STATUS.md").relative_to(ROOT)),
        str((docs_dir / "IMPLEMENTATION_BLOCKERS.md").relative_to(ROOT)),
    ],
    input_row_counts={
        row["file"]: (full_row_counts.get(row["file"], row["sample_rows"]) if not SMOKE else row["sample_rows"])
        for row in inventory_rows
    },
    output_row_counts={"raw_file_inventory.csv": inventory.height, "missing_required_fields.csv": missing_required.height},
    primary_key=["table"],
    duplicate_key_count=int(inventory.select("table").is_duplicated().sum()),
    excluded_row_count=0,
    exception_file=str(missing_path.relative_to(ROOT)),
    status="passed" if blocking_count == 0 else "blocked",
    notes=["All checks used real local inputs; raw CSV files were not modified."],
)
print(summary_path)
